# Machine Translation using Transformers

In [1]:
import os.path
from typing import Any

import torch
import numpy as np
import spacy
import pandas as pd
import sklearn
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import os

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.mps.is_available() else "cpu"
)

## English to Hindi

In [2]:
dir_name = "indic_languages_corpus/bilingual/hi-en"

In [3]:
english = None
with open(os.path.join(os.path.abspath(dir_name), "train.en")) as f:
    english = f.readlines()

In [4]:
hindi = None
with open(os.path.join(os.path.abspath(dir_name), "train.hi")) as f:
    hindi = f.readlines()

In [5]:
X_train = english

In [6]:
y_train = hindi

##### Why not byte pair encoding or character by character or turn words into graph for autocomplete, or sentence translation or generation?

**We are designing infinte window transformers, handling millions or billions of sequences, we can easily do this**
###### Can we turn the data into graphs, since relationship between words is sparse same as wordnet

### Preprocessing

In [7]:
from collections import defaultdict, Counter

In [8]:
sub_corpus = X_train[:2]


def tokenize(text):
    vocabulary = Counter()
    corpus = [a.split() for a in text]

    for token in corpus:
        vocabulary.update([t for t in token])
    word_to_index = {word: i + 1 for i, (word, _) in enumerate(vocabulary.items())}
    word_to_index["pad"] = 0
    numerical_sequences = [
        [word_to_index[token] for token in tokens] for tokens in corpus
    ]
    max_length = max(len(seq) for seq in numerical_sequences)

    padded_sequences = [
        seq + [word_to_index["pad"]] * (max_length - len(seq))
        for seq in numerical_sequences
    ]
    return padded_sequences, vocabulary, word_to_index

In [9]:
padded_sequences, vocabulary_train, train_index = tokenize(X_train)

In [10]:
train_sequences, vocabulary, word_index_train = (
    padded_sequences,
    vocabulary_train,
    train_index,
)

In [11]:
target_sequences, vocabulary_target, target_index = tokenize(y_train)

In [12]:
test_sequences, vocabulary_test, word_index_test = (
    target_sequences,
    vocabulary_target,
    target_index,
)

In [13]:
len(train_sequences)

84557

In [14]:
len(vocabulary_train)

49762

In [15]:
len(vocabulary_test)

40396

In [16]:
vocabulary_train["pad"] = 0
vocabulary_target["pad"] = 0

In [17]:
vocabulary = vocabulary_train
vocabulary_test = vocabulary_target

In [18]:
len(word_index_test)

40397

In [19]:
# vocabulary_target.most_common(40)

In [20]:
vocabulary.most_common(10)

[('the', 14550),
 ('I', 12912),
 ('to', 11695),
 ('you', 11340),
 ('a', 9463),
 ('-', 8610),
 ('of', 5849),
 ('is', 4615),
 ('and', 4566),
 ('in', 4541)]

In [21]:
torch.mps.is_available()

False

In [22]:
X_train[:2]

['And what is their Sigil?\n', 'I do not want to die.\n']

In [23]:
y_train[:2]

['और उनके Sigil क्या है?\n', 'मैं मरना नहीं चाहता.\n']

In [24]:
train_sequences[:2]

[[1,
  2,
  3,
  4,
  5,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 [6,
  7,
  8,
  9,
  10,
  11,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0]]

In [25]:
len(vocabulary_train), len(vocabulary_test), len(word_index_train), len(word_index_test)

(49763, 40397, 49763, 40397)

In [26]:
train_sequences[:2]

[[1,
  2,
  3,
  4,
  5,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 [6,
  7,
  8,
  9,
  10,
  11,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0]]

In [27]:
VOCAB_SIZE = len(vocabulary)
BATCH_SIZE = 64

embedding_dim = 256
# no of GRUs
units = 1024
vocab_inp_size = len(train_index)
vocabulary_target_size = len(target_index)

In [28]:
VOCAB_SIZE

49763

In [29]:
X_train = torch.tensor(train_sequences, dtype=torch.long)
y_train = torch.tensor(target_sequences, dtype=torch.long)

In [30]:
# import pandas as pd
# df = pd.DataFrame.from_dict({'sentences':corpus,'numerical':numerical_sequences})

In [31]:
# df

### Encoder Decoder Architecture

In [32]:
class AttentionHead(nn.Module):
    """
    Multi Attention Head
    Splits Input in Q,K,V

    """

    def __init__(self, model_dim, H, dropout_rate=0.1):
        super().__init__()
        self.Wq = nn.Linear(model_dim, model_dim)
        self.Wk = nn.Linear(model_dim, model_dim)
        self.Wv = nn.Linear(model_dim, model_dim)
        self.H = H
        self.d_h = int(model_dim / H)
        self.dropout = nn.Dropout(p=dropout_rate)

        self.Wo = nn.Linear(model_dim, model_dim)

    def forward(self, sequences, attn_mask=False):
        """Input shape: [batch_size, seq_len, d_model=num_head * d_head]
        # if key_value_states are provided this layer is used as a cross-attention layer for text translation..

        # for the decoder

        """
        batch_size, seq_len, model_dim = sequences.size()
        Q = self.Wq(sequences)

        K, V = self.Wk(sequences), self.Wv(sequences)

        A = Q @ K.transpose(-2, -1)
        if attn_mask is not None and attn_mask:
            A = A.masked_fill(attn_mask == 0, -float("inf"))
        A = F.softmax(A / self.d_h**0.5, dim=-1)  # Applying softmax
        A = self.dropout(A)  # Final

        # Output Z

        Z = A @ V  # torch,tensor
        print(f"{Z.shape=}")
        ### Concatenating in parallel along heads and sequences
        ## Because continuous input
        # Z = (Z.contiguous().view(
        #     batch_size,seq_len,self.H*self.d_h)
        # )
        # 2. Transpose to move seq_len before H: shape (batch_size, seq_len, H, d_h)
        # NOTE: Z is now NON-CONTIGUOUS in memory!
        Z = Z.transpose(1, 2).reshape(batch_size, seq_len, model_dim)
        # 3. Concatenate all heads into d_model = H * d_h
        # Fails without .contiguous():
        # Z = Z.contiguous().view(batch_size,seq_len,self.H*self.d_h)
        # final linear projections
        Z = self.Wo(Z)
        return A, Z

In [33]:
vocabulary["pad"] = 0

### Debug

In [34]:
batch_size = 64
sequence_length = 200
model_dim = 2048
d_model = 512
d_ff = 2048
H = 8
dropout = 0.1

In [35]:
model = AttentionHead(d_model, H)

In [36]:
len(padded_sequences[0])
dim = len(padded_sequences[0])

In [37]:
model

AttentionHead(
  (Wq): Linear(in_features=512, out_features=512, bias=True)
  (Wk): Linear(in_features=512, out_features=512, bias=True)
  (Wv): Linear(in_features=512, out_features=512, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (Wo): Linear(in_features=512, out_features=512, bias=True)
)

In [38]:
x = torch.randn(batch_size, sequence_length, d_model)

In [39]:
Z = model(x)

Z.shape=torch.Size([64, 200, 512])


In [40]:
# len(long_tokenized)

In [41]:
len(vocabulary)

49763

In [42]:
# long_tokenized[:2].shape

In [43]:
# long_tokenized.shape

In [44]:
from sklearn.model_selection import train_test_split

In [45]:
# len(vocabulary.values())

## Feed Forward and Attention Residual

In [46]:
batch, sentence_length, embedding_dim = 20, 10, 8
embedding = torch.randn(batch, sentence_length, embedding_dim)
layer_norm = nn.LayerNorm(embedding_dim)

In [47]:
x = layer_norm(embedding)

In [48]:
x

tensor([[[ 0.7723, -1.0241,  0.9095,  ...,  0.3216, -0.8523, -0.1347],
         [ 1.6535, -0.2120,  0.8935,  ..., -0.1710,  0.1242, -1.7948],
         [ 1.5542, -1.5395,  0.5255,  ..., -1.2223, -0.2635,  0.1544],
         ...,
         [-1.3384, -0.4631, -0.1437,  ...,  0.7264,  1.6701,  1.1589],
         [ 0.6770, -2.1750,  0.9450,  ..., -0.1433,  1.2067, -0.6026],
         [ 0.0764,  0.1443, -1.8865,  ...,  1.3298,  0.7983,  0.0640]],

        [[-0.6467, -0.7087, -2.0276,  ...,  0.3796,  1.3381,  0.3069],
         [-0.2167,  0.7959, -0.0971,  ..., -1.7131,  1.1458,  1.4474],
         [-0.1634, -1.9123,  1.3191,  ...,  0.1104, -0.2216, -0.9154],
         ...,
         [-1.9717, -0.0355,  0.2402,  ...,  0.5679,  1.5983, -0.9063],
         [ 0.2174, -1.3541, -0.2095,  ...,  0.5440,  0.7743, -1.6360],
         [ 1.4101, -0.2961, -0.3447,  ...,  0.1553,  0.9763,  0.6109]],

        [[ 1.1463, -0.8695, -0.9739,  ...,  1.1993,  0.6560, -0.8237],
         [ 0.9056,  1.8676, -1.3169,  ..., -0

In [49]:
embedding = nn.Embedding(10, 3, padding_idx=0)

In [50]:
embedding_sample = nn.Embedding(10, 3, padding_idx=0)

In [51]:
input = torch.LongTensor([[0, 2, 0, 5]])

In [52]:
input.shape

torch.Size([1, 4])

In [53]:
embedding_sample = embedding_sample(input)

In [54]:
embedding_sample

tensor([[[ 0.0000,  0.0000,  0.0000],
         [-1.0758,  0.0908, -0.2195],
         [ 0.0000,  0.0000,  0.0000],
         [-1.0820,  2.4615, -1.1675]]], grad_fn=<EmbeddingBackward0>)

In [55]:
class FFN(nn.Sequential):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.layer1 = nn.Linear(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_ff)
        self.activation = nn.GELU()
        self.out = nn.Linear(d_ff, d_model)

    def forward(self, x):
        x = self.layer1(x)
        x = self.norm1(x)
        x = self.activation(x)
        x = self.out(x)
        return x

In [56]:
ffn = FFN(d_model, d_ff)

In [57]:
ffn

FFN(
  (layer1): Linear(in_features=512, out_features=2048, bias=True)
  (norm1): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
  (activation): GELU(approximate='none')
  (out): Linear(in_features=2048, out_features=512, bias=True)
)

In [58]:
ffn = nn.Sequential(
    nn.Linear(d_model, d_ff), nn.LayerNorm(d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
)

In [59]:
x.shape, d_ff, d_model

(torch.Size([20, 10, 8]), 2048, 512)

In [60]:
class Transformer(nn.Module):
    def __init__(self, d_model, d_ff, num_head, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.d_ff = d_ff
        self.H = num_head
        self.dropout = nn.Dropout(p=dropout)

        self.attention = AttentionHead(d_model, num_head, dropout)
        self.ffn = ffn(d_model, d_ff)

    def forward(self, x, attn_mask=False):
        Z = self.attention(x, attn_mask)
        Z = self.dropout(Z)
        Z = Z + x
        Z = self.ffn(Z) + x
        return Z

In [61]:
corpus = X_train

In [62]:
embedding = nn.Embedding(10, 3, padding_idx=0)
input = torch.LongTensor([[0, 2, 0, 5]])
embedding(input).shape

torch.Size([1, 4, 3])

## Encoder Decoder

In [63]:
d_in = len(corpus)

In [64]:
# encoder = nn.Linear(long_tokenized.shape[0],long_tokenized.shape[1])

In [65]:
embedding

Embedding(10, 3, padding_idx=0)

### Constants

In [66]:
VOCAB_SIZE = len(vocabulary)
BATCH_SIZE = 64

embedding_dim = 256
# no of GRUs
units = 1024
EPOCHS = 30
vocab_inp_size = len(vocabulary)
vocabulary_target_size = len(vocabulary_target)

In [67]:
len(train_index)

49763

### Data

In [68]:
from torch.utils.data import Dataset, DataLoader

In [69]:
dataset = TensorDataset(X_train, y_train)

In [70]:
X_train[0]

tensor([1, 2, 3, 4, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0])

In [71]:
dataset[0]

(tensor([1, 2, 3, 4, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]),
 tensor([1, 2, 3, 4, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]))

In [72]:
# todo convert to graph
class Hindi_English(Dataset):
    def __init__(self):
        pass

In [73]:
data_loader = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=2)

In [74]:
x, y = next(iter(data_loader))

In [75]:
len(data_loader)

1322

### Gru Operation

In [76]:
n, d, m = 3, 5, 7
embedding = nn.Embedding(n, d, max_norm=1.0)
print(f"{embedding.weight.shape=}")
W = torch.randn((m, d), requires_grad=True)
print(f"{W.shape=}")
idx = torch.tensor([1, 2])
a = (
    embedding.weight.clone() @ W.t()
)  # weight must be cloned for this to be differentiable
print(f"{a.shape=},{(a.unsqueeze(0)).shape=}")
b = embedding(idx) @ W.t()  # modifies weight in-place
print(f"{b.shape},{(b.unsqueeze(1)).shape=}")
out = a.unsqueeze(0) + b.unsqueeze(1)
loss = out.sigmoid().prod()
print(loss.backward())

embedding.weight.shape=torch.Size([3, 5])
W.shape=torch.Size([7, 5])
a.shape=torch.Size([3, 7]),(a.unsqueeze(0)).shape=torch.Size([1, 3, 7])
torch.Size([2, 7]),(b.unsqueeze(1)).shape=torch.Size([2, 1, 7])
None


In [77]:
out.shape

torch.Size([2, 3, 7])

In [79]:
rnn = nn.GRU(input_size=10, hidden_size=20, num_layers=2)
input = torch.randn(2, 3, 10)
#embedding_inp = nn.Embedding()
h0 = torch.randn(2, 3, 20)  # first if batch size or number of layers
output, hn = rnn(input, h0)
print(f"{output.shape=},{h0.shape=}")

output.shape=torch.Size([2, 3, 20]),h0.shape=torch.Size([2, 3, 20])


### Encoder

In [80]:
import numpy as np


class Encoder(nn.Module):
    def __init__(self, batch_sz, embedding_dim, enc_units, vocab_size):
        super().__init__()
        self.batch_sz = batch_sz  # set batch size
        self.enc_units = enc_units  # set the number of GRU units
        self.embedding_layer = nn.Embedding(vocab_size, embedding_dim)
        # self.hidden_state=256
        self.gru = nn.GRU(embedding_dim, self.enc_units, batch_first=True)

    def forward(self, x, hidden=None):
        print(f"{x.shape}")
        x = self.embedding_layer(x)
        print(f"Embedded: {x.shape=}")
        # assert h==sample_hidden
        output, state = self.gru(x, hidden)
        return output, state

    def initialize_hidden(self):
        return torch.zeros(1, self.batch_sz, self.enc_units)

### Decoder

In [114]:
class Decoder(nn.Module):
    def __init__(self, batch_sz, embedding_dim, dec_units, vocab_size):
        super().__init__()
        self.batch_sz = batch_sz  # batch_size which is defined as 64
        self.dec_units = dec_units  # the number of decoder GRU units
        self.embedding_layer = nn.Embedding(
            vocab_size,
            embedding_dim,

        )
        self.gru = nn.GRU(embedding_dim, self.dec_units)
        self.out = nn.Linear(dec_units, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embedding_layer(x)
        output, state = self.gru(x, hidden)
        print(f"{output.shape=},{state.shape=}")

        logits = self.out(output)
        return logits, state

### Debug


In [115]:
encoder = Encoder(BATCH_SIZE, embedding_dim, units, vocab_inp_size)

In [116]:
encoder

Encoder(
  (embedding_layer): Embedding(49763, 256)
  (gru): GRU(256, 1024, batch_first=True)
)

In [117]:
sample_hidden = encoder.initialize_hidden()

In [118]:
print(f"f{sample_hidden.shape=}")

fsample_hidden.shape=torch.Size([1, 64, 1024])


In [119]:
65536 / 64 / 16

64.0

In [120]:
out, state = encoder(x)
print(f"{out.shape=},{state.shape=}")

torch.Size([64, 56])
Embedded: x.shape=torch.Size([64, 56, 256])
out.shape=torch.Size([64, 56, 1024]),state.shape=torch.Size([1, 64, 1024])


In [121]:
x.shape

torch.Size([64, 56])

In [122]:
out1, state1 = encoder(x, sample_hidden)
print(f"{out1.shape=},{state1.shape=}")

torch.Size([64, 56])
Embedded: x.shape=torch.Size([64, 56, 256])
out1.shape=torch.Size([64, 56, 1024]),state1.shape=torch.Size([1, 64, 1024])


In [123]:
decoder = Decoder(BATCH_SIZE, 1024, units, len(vocabulary_target))

In [124]:
sample_hidden = encoder.initialize_hidden()

In [125]:
sample_hidden.shape

torch.Size([1, 64, 1024])

In [126]:
x.shape

torch.Size([64, 56])

In [127]:
import numpy as np

uniform = np.random.uniform(1, (BATCH_SIZE, 1))
print(f"{uniform.shape=}")
x1 = torch.rand(BATCH_SIZE, 2).uniform_(-3, 3)
x1

uniform.shape=(2,)


tensor([[-2.4659,  1.6119],
        [ 1.7910, -1.2327],
        [ 0.8137, -0.7173],
        [-2.2586,  1.3644],
        [-2.4907,  0.5381],
        [ 1.9680, -0.5613],
        [-0.3397, -2.0878],
        [-1.3116,  1.8170],
        [-2.5206, -2.5215],
        [ 1.6113,  2.2015],
        [-0.7916, -1.8341],
        [ 2.8602,  1.5066],
        [ 2.7974,  2.3527],
        [-2.7533,  2.4822],
        [ 0.8868,  0.8069],
        [-1.0127, -1.0199],
        [-0.6098,  2.8480],
        [-1.7639, -2.1247],
        [-0.7223,  1.5681],
        [-0.7176, -2.1560],
        [-2.5907, -1.7592],
        [ 1.7722,  1.4819],
        [-1.5887,  0.6948],
        [-0.5392, -0.8365],
        [-1.4736, -1.0772],
        [ 0.0099,  2.1992],
        [-0.7353,  2.2135],
        [ 2.3522,  0.3646],
        [ 2.2383, -0.9788],
        [ 1.6647,  2.9034],
        [-2.0503, -1.4447],
        [-1.8596, -1.9975],
        [-2.6768, -1.6684],
        [-1.4923,  2.1137],
        [ 1.9933, -2.6493],
        [-2.7258,  1

In [128]:
uniform.ndim

1

In [129]:
encoder

Encoder(
  (embedding_layer): Embedding(49763, 256)
  (gru): GRU(256, 1024, batch_first=True)
)

In [130]:
list(encoder.named_modules())

[('',
  Encoder(
    (embedding_layer): Embedding(49763, 256)
    (gru): GRU(256, 1024, batch_first=True)
  )),
 ('embedding_layer', Embedding(49763, 256)),
 ('gru', GRU(256, 1024, batch_first=True))]

In [131]:
x1.ndim

2

In [132]:
x1 = torch.tensor(x1.detach().clone(), dtype=torch.long)

/var/folders/px/m0g9wbyn1sv678fsx9lgsm600000gn/T/ipykernel_25003/3818009117.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x1 = torch.tensor(x1.detach().clone(), dtype=torch.long)


In [133]:
x1.ndim

2

In [134]:
sample_decoder_output, state2 = decoder(torch.tensor(uniform, dtype=torch.long))
print(sample_decoder_output.shape, state2.shape)

output.shape=torch.Size([2, 1024]),state.shape=torch.Size([1, 1024])
torch.Size([2, 40397]) torch.Size([1, 1024])


In [135]:
decoder

Decoder(
  (embedding_layer): Embedding(40397, 1024)
  (gru): GRU(1024, 1024)
  (out): Linear(in_features=1024, out_features=40397, bias=True)
)

In [137]:
list(decoder.state_dict().keys())

['embedding_layer.weight',
 'gru.weight_ih_l0',
 'gru.weight_hh_l0',
 'gru.bias_ih_l0',
 'gru.bias_hh_l0',
 'out.weight',
 'out.bias']

In [139]:
sample_decoder_output2, state2 = decoder(
    torch.tensor(np.random.uniform(3, (BATCH_SIZE, 3)), dtype=torch.long),
)
sample_decoder_output2

output.shape=torch.Size([2, 1024]),state.shape=torch.Size([1, 1024])


tensor([[-0.0764, -0.2132, -0.1000,  ...,  0.0529,  0.1691, -0.0989],
        [-0.1789, -0.1439, -0.1014,  ...,  0.0240,  0.1971, -0.1708]],
       grad_fn=<AddmmBackward0>)

In [140]:
print(sample_decoder_output2.shape, state2.shape)

torch.Size([2, 40397]) torch.Size([1, 1024])


In [141]:
sample_decoder_output1, state1 = decoder(torch.tensor(uniform, dtype=torch.long))

output.shape=torch.Size([2, 1024]),state.shape=torch.Size([1, 1024])


In [142]:
len(data_loader)

1322

In [143]:
x

tensor([[  1,   2,   3,  ...,   0,   0,   0],
        [  6,   7,   8,  ...,   0,   0,   0],
        [ 12,  13,  14,  ...,   0,   0,   0],
        ...,
        [256,  61, 257,  ...,   0,   0,   0],
        [259,  13, 260,  ...,   0,   0,   0],
        [261, 262,   0,  ...,   0,   0,   0]])

In [144]:
y

tensor([[  1,   2,   3,  ...,   0,   0,   0],
        [  6,   7,   8,  ...,   0,   0,   0],
        [ 10,  11,  12,  ...,   0,   0,   0],
        ...,
        [248, 166, 180,  ...,   0,   0,   0],
        [250,  36, 251,  ...,   0,   0,   0],
        [252, 253,   0,  ...,   0,   0,   0]])

In [145]:
sample_hidden = encoder.initialize_hidden()

In [146]:
sample_hidden.shape

torch.Size([1, 64, 1024])

In [155]:
sample_hidden = sample_hidden.view(1, sample_hidden.shape[-1], sample_hidden.shape[-2])

In [156]:
sample_hidden.shape

torch.Size([1, 1024, 64])

In [157]:
x.shape

torch.Size([64, 56])

In [158]:
x_seq = torch.tensor([[1.0] * 5, [2.0] * 5, [3.0] * 5])
x_seq.shape

torch.Size([3, 5])

In [159]:
x_seq.reshape(1, 3, 5)

tensor([[[1., 1., 1., 1., 1.],
         [2., 2., 2., 2., 2.],
         [3., 3., 3., 3., 3.]]])

In [160]:
x.shape

torch.Size([64, 56])

In [161]:
batch_size

64

In [162]:
x_batched = x.view(batch_size, 1, x.shape[-1])

In [163]:
x_batched.shape

torch.Size([64, 1, 56])

In [164]:
X_train.shape

torch.Size([84557, 56])

In [ ]:
# sample_output, sample_hidden = encoder(x, sample_hidden)

In [ ]:
import dataclasses
from dataclasses import dataclass

from dataclasses import dataclass


@dataclass(init=True, repr=True)
class InventoryItem:
    """Class for keeping track of an item in inventory."""

    name: str
    unit_price: float
    quantity_on_hand: int = 0

    def total_cost(self) -> float:
        return self.unit_price * self.quantity_on_hand


item = InventoryItem(name="T shirt", unit_price=10, quantity_on_hand=3)
print(item, item.total_cost())

In [ ]:
input = torch.randn(3, 2, requires_grad=True)
target = torch.rand(3, 2, requires_grad=False)
loss = F.binary_cross_entropy(torch.sigmoid(input), target)
loss.backward()
print(loss.item())

In [ ]:
batch_size

In [ ]:
encoder

## Training

In [166]:
from typing import List
from dataclasses import dataclass

@dataclass(init=True)
class Result:
    train_loss: List[float]
    test_loss: List[float]

    train_acc: List[float]
    test_acc: List[float]

In [167]:
dir(encoder)

['T_destination',
 '__annotate_func__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_apply',
 '_backward_hooks',
 '_backward_pre_hooks',
 '_buffers',
 '_call_impl',
 '_compiled_call_impl',
 '_forward_hooks',
 '_forward_hooks_always_called',
 '_forward_hooks_with_kwargs',
 '_forward_pre_hooks',
 '_forward_pre_hooks_with_kwargs',
 '_get_backward_hooks',
 '_get_backward_pre_hooks',
 '_get_name',
 '_is_full_backward_hook',
 '_load_from_state_dict',
 '_load_state_dict_post_hooks',
 '_load_state_dict_pre_hooks',
 '_maybe_warn_non_full_backward_hook',
 '_modules',

In [168]:
encoder = encoder.to(device)

In [169]:
decoder = decoder.to(device)

In [170]:
loss = torch.nn.BCEWithLogitsLoss()
criterion = torch.optim.AdamW(encoder.parameters(), lr=0.3, weight_decay=0.1)

In [171]:
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

In [172]:
for p, m in encoder.named_parameters():
    print(f"{p},{m.shape=}")

embedding_layer.weight,m.shape=torch.Size([49763, 256])
gru.weight_ih_l0,m.shape=torch.Size([3072, 256])
gru.weight_hh_l0,m.shape=torch.Size([3072, 1024])
gru.bias_ih_l0,m.shape=torch.Size([3072])
gru.bias_hh_l0,m.shape=torch.Size([3072])


In [173]:
loss_object = torch.nn.BCEWithLogitsLoss()

In [174]:
len(loader)

1321

In [175]:
@torch.compile(options={"triton.cudagraphs": False}, fullgraph=True)
def foo(x):
    return torch.sin(x) + torch.cos(x)

In [176]:
foo(torch.tensor([2]))

tensor([0.4932])

In [240]:
from tqdm.auto import tqdm
@torch.compile
def train_epoch(epoch: int,model: nn.Module,train_loader: torch.utils.data.DataLoader,optim: torch.optim.Optimizer):
    model.train()

    criterion = nn.CrossEntropyLoss()
    for batched_input, batched_target in tqdm(
        train_loader, desc=f"Training @ epoch {epoch}"
    ):
      batched_input,batched_target   = batched_input.to(device),batched_target.to(device)
      optim.zero_grad()
      loss = criterion(model(batched_input), batched_targ,



